## Gold — fact tables.
 Two rules govern every fact here:

   1. Patient identity resolves through silver_patient_xref, never through
      a source system's own id. This is what makes cross-facility
      readmission analysis possible.

   2. A lookup that fails points at the unknown member (-1), never drops
      the row. Silver-to-Gold counts then reconcile, and the gap is a
      measurable quantity rather than a silent undercount.

 Facts built:
   -fact_admission            accumulating snapshot, LOS + readmission
   - fact_emergency_visit      accumulating snapshot, wait-time milestones
   - fact_appointment          transaction
   - fact_lab_result           transaction
   - fact_medication_order     transaction
   - fact_claim                accumulating snapshot
   - fact_billing_line         transaction
   - fact_satisfaction_survey  transaction
   - fact_bed_occupancy_daily  periodic snapshot, bed x day

In [ ]:
batch_id = "GOLDFACT_MANUAL"
SILVER = "lh_silver.dbo"
READMISSION_WINDOW_DAYS = 30
occupancy_days = 365          # rolling window for the bed snapshot

In [ ]:
from datetime import datetime, timezone

from pyspark.sql import functions as F, Window

run_ts = datetime.now(timezone.utc)
UNKNOWN = -1
summary = []

print(f"Gold fact load {batch_id}")

## Helpers


In [ ]:
def date_key(col):
    return F.when(col.isNull(), F.lit(UNKNOWN)) \
            .otherwise(F.date_format(col, "yyyyMMdd").cast("int"))


def time_key(col):
    return F.when(col.isNull(), F.lit(UNKNOWN)) \
            .otherwise((F.hour(col) * 60 + F.minute(col)).cast("int"))


def lookup(fact, dim_table, fact_col, dim_bk, dim_key, out=None, event_ts=None):
    """Resolve a surrogate key, defaulting to the unknown member.

    dim_key is the column read FROM the dimension; out is the name written
    onto the fact. They differ whenever one dimension serves several roles —
    dim_diagnosis provides both primary_diagnosis_key and
    triage_diagnosis_key from the same diagnosis_key column.

    When event_ts is supplied the lookup is temporally correct against an
    SCD2 dimension: the version of the member in effect when the event
    happened, not today's version. Using the current version would
    attribute a 2024 admission to a doctor's 2026 department.
    """
    out = out or dim_key
    dim = spark.table(dim_table)

    if event_ts and "effective_from_ts" in dim.columns:
        d = dim.select(F.col(dim_bk).alias("_bk"), F.col(dim_key).alias("_sk"),
                       F.col("effective_from_ts").alias("_from"),
                       F.col("effective_to_ts").alias("_to"))
        cond = ((fact[fact_col] == F.col("_bk"))
                & (fact[event_ts] >= F.col("_from"))
                & (fact[event_ts] < F.col("_to")))
    else:
        d = dim
        if "is_current" in dim.columns:
            d = d.filter(F.col("is_current"))
        d = d.select(F.col(dim_bk).alias("_bk"), F.col(dim_key).alias("_sk"))
        cond = fact[fact_col] == F.col("_bk")

    return (fact.join(d, cond, "left")
                .withColumn(out, F.coalesce(F.col("_sk"), F.lit(UNKNOWN)))
                .drop("_bk", "_sk", "_from", "_to"))


def finish(df, table, source_rows):
    df = (df.withColumn("batch_id", F.lit(batch_id))
            .withColumn("loaded_ts", F.lit(run_ts)))
    df.write.format("delta").mode("overwrite") \
      .option("overwriteSchema", "true").saveAsTable(table)
    n = spark.table(table).count()
    summary.append((table, source_rows, n))
    print(f"  {table:<30} silver={source_rows:>9,}  gold={n:>9,}")


def unknown_rate(table, key_cols):
    df = spark.table(table)
    total = df.count()
    out = []
    for k in key_cols:
        if k in df.columns:
            n = df.filter(F.col(k) == UNKNOWN).count()
            if n:
                out.append(f"{k}={100*n/max(1,total):.1f}%")
    return ", ".join(out) if out else "all keys resolved"


## 1. fact_admission

 Accumulating snapshot: the row is created on admission and revised on
 discharge, so open admissions are visible in the model rather than
 appearing only after the patient leaves.

In [ ]:
adm = spark.table(f"{SILVER}.silver_admission")
n_src = adm.count()

# Bed movements: transfer count, ICU days, and the admitting bed.
moves = (spark.table(f"{SILVER}.silver_bed_assignment")
         .join(spark.table("dim_bed").filter("is_current")
               .select(F.col("bed_id").alias("_bid"), "bed_key", "ward_type"),
               F.col("bed_id") == F.col("_bid"), "left"))

move_agg = moves.groupBy("admission_id").agg(
    F.greatest(F.count("*") - 1, F.lit(0)).alias("transfer_count"),
    F.sum(F.when(F.col("ward_type") == "ICU",
                 (F.col("assignment_end_ts").cast("long")
                  - F.col("assignment_start_ts").cast("long")) / 86400.0)
           .otherwise(F.lit(0.0))).alias("icu_days"))

first_bed = (moves.withColumn("_rn", F.row_number().over(
                 Window.partitionBy("admission_id").orderBy("assignment_start_ts")))
             .filter("_rn = 1").select("admission_id",
                                       F.col("bed_key").alias("first_bed_key")))

# Readmission. Each exclusion is applied explicitly so the definition can be
# audited against the ministry specification rather than inferred from code.
disp = spark.table("dim_discharge_disposition").select(
    F.col("disposition_code").alias("_dc"), "is_transfer", "is_expired")

f = adm.join(disp, adm["discharge_disposition_std"] == F.col("_dc"), "left")

w = Window.partitionBy("patient_golden_id").orderBy("admission_ts")
f = (f
    .withColumn("_prior_discharge", F.lag("discharge_ts").over(w))
    .withColumn("_prior_transfer",
                F.coalesce(F.lag("is_transfer").over(w), F.lit(False)))
    .withColumn("_prior_death",
                F.coalesce(F.lag("is_expired").over(w), F.lit(False)))
    .withColumn("days_since_prior_discharge",
                F.when(F.col("_prior_discharge").isNotNull(),
                       F.datediff("admission_ts", "_prior_discharge")))
    .withColumn("is_readmission_30d", F.coalesce(
        F.col("days_since_prior_discharge").between(0, READMISSION_WINDOW_DAYS)
        & (F.col("admission_type_std") != "ELECTIVE")
        & (~F.col("_prior_transfer"))
        & (~F.col("_prior_death")), F.lit(False)))
    .withColumn("is_index_admission", ~F.col("is_readmission_30d")))

f = lookup(f, "dim_patient", "patient_golden_id", "patient_golden_id",
           "patient_key", event_ts="admission_ts")
f = lookup(f, "dim_doctor", "attending_physician_id", "doctor_id",
           "doctor_key", out="attending_doctor_key", event_ts="admission_ts")
f = lookup(f, "dim_department", "department_id", "department_id",
           "department_key", event_ts="admission_ts")
f = lookup(f, "dim_hospital", "hospital_id", "hospital_id", "hospital_key",
           event_ts="admission_ts")
f = lookup(f, "dim_admission_type", "admission_type_std",
           "admission_type_code", "admission_type_key")
f = lookup(f, "dim_discharge_disposition", "discharge_disposition_std",
           "disposition_code", "discharge_disposition_key")
f = lookup(f, "dim_diagnosis", "primary_diagnosis_code", "diagnosis_code",
           "diagnosis_key", out="primary_diagnosis_key")
f = lookup(f, "dim_insurance_provider", "payer_id", "payer_id",
           "insurance_provider_key")

f = f.join(move_agg, "admission_id", "left").join(first_bed, "admission_id", "left")

charges = (spark.table(f"{SILVER}.silver_billing_line")
           .groupBy("encounter_id").agg(F.sum("net_amount").alias("total_charges")))
f = f.join(charges, "encounter_id", "left")

finish(f.select(
    "admission_id", "encounter_id", "patient_key", "attending_doctor_key",
    "department_key", "hospital_key",
    F.coalesce("first_bed_key", F.lit(UNKNOWN)).alias("bed_key"),
    "admission_type_key", "discharge_disposition_key", "primary_diagnosis_key",
    "insurance_provider_key",
    date_key(F.col("admission_ts")).alias("admission_date_key"),
    time_key(F.col("admission_ts")).alias("admission_time_key"),
    date_key(F.col("discharge_ts")).alias("discharge_date_key"),
    date_key(F.col("expected_discharge_ts")).alias("expected_discharge_date_key"),
    "length_of_stay_days", "length_of_stay_hours",
    F.coalesce("icu_days", F.lit(0.0)).alias("icu_days"),
    F.coalesce("transfer_count", F.lit(0)).alias("transfer_count"),
    "days_since_prior_discharge",
    F.coalesce("total_charges", F.lit(0.0)).alias("total_charges"),
    "is_readmission_30d", "is_index_admission", "is_open",
), "fact_admission", n_src)

## 2. fact_emergency_visit


In [ ]:
ed = spark.table(f"{SILVER}.silver_emergency_visit")
n_src = ed.count()

f = lookup(ed, "dim_patient", "patient_golden_id", "patient_golden_id",
           "patient_key", event_ts="arrival_ts")
f = lookup(f, "dim_hospital", "hospital_id", "hospital_id",
           "hospital_key", event_ts="arrival_ts")
f = lookup(f, "dim_department", "department_id", "department_id",
           "department_key", event_ts="arrival_ts")
f = lookup(f, "dim_diagnosis", "triage_diagnosis_code", "diagnosis_code",
           "diagnosis_key", out="triage_diagnosis_key")
f = lookup(f, "dim_insurance_provider", "payer_id", "payer_id",
           "insurance_provider_key")

finish(f.select(
    "ed_visit_id", "encounter_id", "patient_key", "hospital_key",
    "department_key", "triage_diagnosis_key", "insurance_provider_key",
    date_key(F.col("arrival_ts")).alias("arrival_date_key"),
    time_key(F.col("arrival_ts")).alias("arrival_time_key"),
    date_key(F.col("departure_ts")).alias("departure_date_key"),
    "ctas_level", "arrival_mode",
    "triage_wait_min", "physician_initial_assessment_min",
    "decision_to_admit_min", "boarding_min", "total_ed_los_min",
    "left_without_being_seen", "resulted_in_admission",
), "fact_emergency_visit", n_src)

## 3. fact_appointment


In [ ]:
appt = spark.table(f"{SILVER}.silver_appointment")
n_src = appt.count()

f = lookup(appt, "dim_patient", "patient_golden_id", "patient_golden_id",
           "patient_key", event_ts="scheduled_ts")
f = lookup(f, "dim_doctor", "doctor_id", "doctor_id", "doctor_key",
           event_ts="scheduled_ts")
f = lookup(f, "dim_department", "department_id", "department_id",
           "department_key", event_ts="scheduled_ts")
f = lookup(f, "dim_hospital", "hospital_id", "hospital_id", "hospital_key",
           event_ts="scheduled_ts")
f = lookup(f, "dim_appointment_status", "status_code", "status_code",
           "appointment_status_key")

finish(f.select(
    "appointment_id", "patient_key", "doctor_key", "department_key",
    "hospital_key", "appointment_status_key",
    date_key(F.col("booking_ts")).alias("booking_date_key"),
    date_key(F.col("scheduled_ts")).alias("scheduled_date_key"),
    time_key(F.col("scheduled_ts")).alias("scheduled_time_key"),
    "appointment_type", "is_first_visit", "is_virtual",
    "scheduled_duration_min", "actual_duration_min", "wait_in_clinic_min",
    "lead_time_days", "cancellation_notice_hours", "cancelled_by",
    "is_completed", "is_cancelled", "is_no_show",
), "fact_appointment", n_src)

## 4. fact_lab_result


In [ ]:
lab = spark.table(f"{SILVER}.silver_lab_result")
n_src = lab.count()

f = lookup(lab, "dim_patient", "patient_golden_id", "patient_golden_id",
           "patient_key", event_ts="order_ts")
f = lookup(f, "dim_doctor", "ordering_doctor_id", "doctor_id",
           "doctor_key", out="ordering_doctor_key", event_ts="order_ts")
f = lookup(f, "dim_department", "department_id", "department_id",
           "department_key", event_ts="order_ts")
f = lookup(f, "dim_hospital", "sending_facility", "hospital_id", "hospital_key",
           event_ts="order_ts")
f = lookup(f, "dim_lab_test", "loinc_code", "loinc_code", "lab_test_key")

finish(f.select(
    F.col("placer_order_id").alias("lab_result_id"),
    "encounter_id", "patient_key", "ordering_doctor_key", "department_key",
    "hospital_key", "lab_test_key",
    date_key(F.col("order_ts")).alias("order_date_key"),
    time_key(F.col("order_ts")).alias("order_time_key"),
    date_key(F.col("result_ts")).alias("result_date_key"),
    "priority", "result_value_numeric", "result_value_text", "abnormal_flag",
    "is_abnormal", "is_critical", "total_turnaround_min",
), "fact_lab_result", n_src)

## 5. fact_medication_order


In [ ]:
med = spark.table(f"{SILVER}.silver_medication_order")
n_src = med.count()

f = lookup(med, "dim_patient", "patient_golden_id", "patient_golden_id",
           "patient_key", event_ts="order_ts")
f = lookup(f, "dim_medication", "din", "din", "medication_key")
f = lookup(f, "dim_doctor", "prescribing_doctor_id", "doctor_id",
           "doctor_key", out="prescribing_doctor_key", event_ts="order_ts")
f = lookup(f, "dim_department", "department_id", "department_id",
           "department_key", event_ts="order_ts")
f = lookup(f, "dim_hospital", "hospital_id", "hospital_id", "hospital_key",
           event_ts="order_ts")
f = lookup(f, "dim_medication", "din", "din", "medication_key")

# is_high_alert lives on the medication dimension, not the order. Carrying
# it onto the fact means a formulary reclassification would not retroactively
# change historical orders.
f = f.join(spark.table("dim_medication").select(
    F.col("medication_key").alias("_mk"), "is_high_alert"),
    F.col("medication_key") == F.col("_mk"), "left").drop("_mk")

finish(f.select(
    "medication_order_id", "encounter_id", "patient_key",
    "prescribing_doctor_key", "department_key", "hospital_key",
    "medication_key",
    date_key(F.col("order_ts")).alias("order_date_key"),
    time_key(F.col("order_ts")).alias("order_time_key"),
    date_key(F.col("dispense_ts")).alias("dispense_date_key"),
    "dose_amount", "dose_unit", "frequency_code",
    "quantity_ordered", "quantity_dispensed", "days_supply",
    "unit_cost", "total_cost", "order_to_dispense_min",
    F.coalesce("is_high_alert", F.lit(False)).alias("is_high_alert"),
), "fact_medication_order", n_src)

## 6. fact_claim


In [ ]:
clm = spark.table(f"{SILVER}.silver_claim")
n_src = clm.count()

# The claim feed carries the patient's EHR id in the subscriber field.
xref = spark.table(f"{SILVER}.silver_patient_xref").select(
    "record_uid", "patient_golden_id")
f = (clm.withColumn("_uid", F.concat_ws("|", F.lit("EHR"), F.col("patient_source_id")))
        .join(xref, F.col("_uid") == F.col("record_uid"), "left").drop("_uid", "record_uid"))

f = lookup(f, "dim_patient", "patient_golden_id", "patient_golden_id", "patient_key")
f = lookup(f, "dim_hospital", "hospital_id", "hospital_id", "hospital_key")
f = lookup(f, "dim_insurance_provider", "payer_id", "payer_id",
           "insurance_provider_key")
f = lookup(f, "dim_claim_status", "status_code", "status_code", "claim_status_key")
f = lookup(f, "dim_diagnosis", "primary_diagnosis_code", "diagnosis_code",
           "diagnosis_key", out="primary_diagnosis_key")

finish(f.select(
    F.col("claim_number").alias("claim_id"),
    "patient_key", "hospital_key", "insurance_provider_key",
    "claim_status_key", "primary_diagnosis_key",
    date_key(F.col("service_date")).alias("service_date_key"),
    date_key(F.col("submission_date")).alias("submission_date_key"),
    date_key(F.col("payment_date")).alias("payment_date_key"),
    "billed_amount", "paid_amount", "denied_amount",
    "patient_responsibility", "denial_reason_code", "days_to_payment",
    "line_count", "is_adjudicated", "is_approved", "is_denied",
), "fact_claim", n_src)

## 7. fact_billing_line and fact_satisfaction_survey


In [ ]:
bill = spark.table(f"{SILVER}.silver_billing_line")
n_src = bill.count()

f = lookup(bill, "dim_patient", "patient_golden_id", "patient_golden_id", "patient_key")
f = lookup(f, "dim_department", "department_id", "department_id", "department_key")
f = lookup(f, "dim_hospital", "hospital_id", "hospital_id", "hospital_key")
f = lookup(f, "dim_insurance_provider", "payer_id", "payer_id",
           "insurance_provider_key")

finish(f.select(
    "invoice_id", F.col("invoice_line_number").cast("int").alias("invoice_line_number"),
    "encounter_id", "patient_key", "department_key", "hospital_key",
    "insurance_provider_key",
    "service_code", "service_category", "quantity",
    "charge_amount", "discount_amount", "tax_amount", "net_amount",
    "payment_amount", "outstanding_amount",
), "fact_billing_line", n_src)

srv = spark.table(f"{SILVER}.silver_survey_response")
n_src = srv.count()

f = lookup(srv, "dim_patient", "patient_golden_id", "patient_golden_id", "patient_key")
f = lookup(f, "dim_doctor", "doctor_id", "doctor_id", "doctor_key")
f = lookup(f, "dim_department", "department_id", "department_id", "department_key")
f = lookup(f, "dim_hospital", "hospital_id", "hospital_id", "hospital_key")

finish(f.select(
    "survey_response_id", "encounter_id", "patient_key", "doctor_key",
    "department_key", "hospital_key",
    date_key(F.col("response_date")).alias("response_date_key"),
    date_key(F.col("service_date")).alias("service_date_key"),
    "encounter_type", "overall_score", "wait_time_score",
    "staff_courtesy_score", "cleanliness_score", "communication_score",
    "pain_management_score", "would_recommend_score", "nps_category",
), "fact_satisfaction_survey", n_src)

## 8. bridge_encounter_diagnosis

 An encounter has one primary diagnosis and several comorbidities. Putting
 them all on the fact would either lose the comorbidities or fan out every
 admission measure. A bridge keeps both possible.


In [ ]:
dx = spark.table(f"{SILVER}.silver_diagnosis")
n_src = dx.count()

f = lookup(dx, "dim_diagnosis", "diagnosis_code", "diagnosis_code", "diagnosis_key")
xref = spark.table(f"{SILVER}.silver_patient_xref").select(
    "record_uid", "patient_golden_id")
f = (f.withColumn("_uid", F.concat_ws("|", F.lit("EHR"), F.col("patient_id")))
      .join(xref, F.col("_uid") == F.col("record_uid"), "left")
      .drop("_uid", "record_uid"))
f = lookup(f, "dim_patient", "patient_golden_id", "patient_golden_id", "patient_key")

finish(f.select(
    "encounter_id", "diagnosis_key", "patient_key",
    date_key(F.col("diagnosis_date")).alias("diagnosis_date_key"),
    "diagnosis_rank", "diagnosis_type", "is_primary", "is_present_on_admission",
), "bridge_encounter_diagnosis", n_src)

## 9. fact_bed_occupancy_daily

 Periodic snapshot at bed x day grain. Built by cross-joining beds with the
 calendar so unoccupied beds produce rows with zero occupancy rather than
 disappearing — without that, occupancy rate would have no denominator.


In [ ]:
beds = spark.table("dim_bed").filter("is_current AND bed_key > 0").select(
    "bed_key", "bed_id", "hospital_id", "department_id")

cal = (spark.table("dim_date")
       .filter(F.col("date_key") > 0)
       .filter(F.col("calendar_date") >= F.date_sub(F.current_date(), occupancy_days))
       .filter(F.col("calendar_date") <= F.current_date())
       .select("date_key", "calendar_date"))

grid = beds.crossJoin(cal)

asg = (spark.table(f"{SILVER}.silver_bed_assignment")
       .select("bed_id", "assignment_start_ts", "assignment_end_ts")
       .filter(F.col("assignment_start_ts").isNotNull()))

day_start = F.col("calendar_date").cast("timestamp")
day_end = F.date_add(F.col("calendar_date"), 1).cast("timestamp")

occ = (grid.join(asg, "bed_id", "left")
    .withColumn("_ov_start", F.greatest(F.col("assignment_start_ts"), day_start))
    .withColumn("_ov_end", F.least(
        F.coalesce(F.col("assignment_end_ts"), day_end), day_end))
    .withColumn("occupied_hours",
                F.when(F.col("_ov_end") > F.col("_ov_start"),
                       (F.col("_ov_end").cast("long")
                        - F.col("_ov_start").cast("long")) / 3600.0)
                 .otherwise(F.lit(0.0)))
    .withColumn("_midnight",
                (F.col("assignment_start_ts") <= day_start)
                & (F.coalesce(F.col("assignment_end_ts"), day_end) > day_start)))

agg = (occ.groupBy("bed_key", "bed_id", "hospital_id", "department_id",
                   "date_key", "calendar_date")
       .agg(F.least(F.sum("occupied_hours"), F.lit(24.0)).alias("occupied_hours"),
            F.max(F.col("_midnight").cast("int")).alias("_mid"),
            F.sum(F.when(F.col("occupied_hours") > 0, 1).otherwise(0)).alias("turnover_count")))

f = lookup(agg, "dim_hospital", "hospital_id", "hospital_id", "hospital_key")
f = lookup(f, "dim_department", "department_id", "department_id", "department_key")

n_src = grid.count()
finish(f.select(
    "date_key", "bed_key", "hospital_key", "department_key",
    F.col("occupied_hours").cast("decimal(6,2)").alias("occupied_hours"),
    (F.lit(24.0) - F.col("occupied_hours")).cast("decimal(6,2)").alias("available_hours"),
    F.lit(0.0).cast("decimal(6,2)").alias("blocked_hours"),
    (F.col("_mid") == 1).alias("is_occupied_at_midnight"),
    F.col("turnover_count").cast("int").alias("turnover_count"),
), "fact_bed_occupancy_daily", n_src)

## Summary and key-resolution check

 The unknown-key rate is the number to watch. A few percent is normal and
 expected. A large share means a dimension lookup is joining on the wrong
 column, and the facts would still load without error.


In [1]:
print("\n" + "=" * 70)
print(f"{'fact':<32}{'silver':>11}{'gold':>11}")
print("-" * 70)
for t, s, g in summary:
    print(f"{t:<32}{s:>11,}{g:>11,}")
print("=" * 70)

print("\nUnresolved dimension keys:")
for t, keys in [
    ("fact_admission", ["patient_key", "attending_doctor_key", "department_key",
                        "hospital_key", "bed_key", "primary_diagnosis_key"]),
    ("fact_emergency_visit", ["patient_key", "hospital_key", "department_key"]),
    ("fact_appointment", ["patient_key", "doctor_key", "department_key"]),
    ("fact_lab_result", ["patient_key", "lab_test_key", "ordering_doctor_key"]),
    ("fact_medication_order", ["patient_key", "medication_key"]),
    ("fact_claim", ["patient_key", "insurance_provider_key", "claim_status_key"]),
    ("fact_billing_line", ["patient_key", "department_key"]),
    ("fact_satisfaction_survey", ["patient_key", "doctor_key"]),
    ("fact_bed_occupancy_daily", ["bed_key", "hospital_key"]),
]:
    print(f"  {t:<30} {unknown_rate(t, keys)}")

import json
mssparkutils.notebook.exit(json.dumps({
    "batch_id": batch_id, "facts": len(summary),
    "total_rows": sum(g for _, _, g in summary),
}))

StatementMeta(, bf6f2644-78c0-4db8-a973-c163018e4224, 3, Finished, Available, Finished, False)

Gold fact load GOLDFACT_MANUAL
  fact_admission                 silver=   11,568  gold=   11,568
  fact_emergency_visit           silver=   25,037  gold=   25,037
  fact_appointment               silver=  228,833  gold=  228,833
  fact_lab_result                silver=  161,319  gold=  161,319
  fact_medication_order          silver=   97,511  gold=   97,511
  fact_claim                     silver=      711  gold=      711
  fact_billing_line              silver=   52,196  gold=   52,196
  fact_satisfaction_survey       silver=    8,471  gold=    8,471
  bridge_encounter_diagnosis     silver=   37,318  gold=   37,318
  fact_bed_occupancy_daily       silver=   36,966  gold=   36,966

fact                                 silver       gold
----------------------------------------------------------------------
fact_admission                       11,568     11,568
fact_emergency_visit                 25,037     25,037
fact_appointment                    228,833    228,833
fact_lab_result  